<a href="https://colab.research.google.com/github/kb0417/french-ewe-translation-transcription/blob/main/notebooks/07_finetuning_nllb_ewe_to_french.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 — Fine-tuning de NLLB pour la traduction éwé → français

Dans le notebook précédent, nous avons réalisé un fine-tuning léger de NLLB pour la traduction français → éwé.

Dans ce notebook, nous adaptons maintenant NLLB dans l’autre sens : éwé → français.

L’objectif est de compléter le système de traduction bidirectionnel :

- français → éwé ;
- éwé → français.

Nous utilisons la même méthode que précédemment : un fine-tuning léger avec LoRA, afin de rendre l'entraînement possible dans Google Colab sans modifier tous les paramètres du modèle.

In [1]:
# On installe ou met à jour les bibliothèques nécessaires.
# torchao est inclus pour éviter les conflits récents avec PEFT.

!pip install -U transformers datasets evaluate sacrebleu sentencepiece accelerate peft torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.6 MB/s eta 0:00:00


In [2]:
import os
import torch
import pandas as pd
import numpy as np

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

from peft import LoraConfig, get_peft_model, TaskType

import evaluate

In [3]:
import transformers
import peft

print("Transformers version :", transformers.__version__)
print("PEFT version :", peft.__version__)

Transformers version : 5.10.2
PEFT version : 0.19.1


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Appareil utilisé :", device)

if device == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("Attention : sans GPU, le fine-tuning sera très lent.")

Appareil utilisé : cuda
GPU : Tesla T4


In [5]:
from google.colab import drive
drive.mount('/content/drive')

project_dir = "/content/drive/MyDrive/french_ewe_project"

data_dir = os.path.join(project_dir, "data/processed")
models_dir = os.path.join(project_dir, "models")
reports_dir = os.path.join(project_dir, "reports")

os.makedirs(models_dir, exist_ok=True)
os.makedirs(reports_dir, exist_ok=True)

print("Dossier données :", data_dir)
print("Dossier modèles :", models_dir)
print("Dossier rapports :", reports_dir)

Mounted at /content/drive
Dossier données : /content/drive/MyDrive/french_ewe_project/data/processed
Dossier modèles : /content/drive/MyDrive/french_ewe_project/models
Dossier rapports : /content/drive/MyDrive/french_ewe_project/reports


In [6]:
train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))
valid_df = pd.read_csv(os.path.join(data_dir, "valid.csv"))
test_df = pd.read_csv(os.path.join(data_dir, "test.csv"))

print("Train :", train_df.shape)
print("Validation :", valid_df.shape)
print("Test :", test_df.shape)

train_df.head()

Train : (18820, 6)
Validation : (2352, 6)
Test : (2353, 6)


,french,ewe,source,type,french_length,ewe_length
0,"""C'est moins lourd pour les enfants, pour le s...",ele hodzoe na ɖeviwo kple ame bubu geɖewo,https://ellecitoyenne.com/,"Blog, News",14,8
1,"""Il ne nous reste que nous, si nous voulons un...",mía ŋutɔwo dzi ɖeɖe ko wole be míaka ɖo ne m...,https://ellecitoyenne.com/,"Blog, News",15,13
2,"""Pendant quelques années, Histoire et History ...",tsakakatsaka ɖeke magava eme le akɔdada le kam...,https://ellecitoyenne.com/,"Blog, News",47,14
3,"""Ainsi, à travers une lecture littérale de la ...",Aleae to bibla xexle me la miedoa dzesi be duk...,https://ellecitoyenne.com/,"Blog, News",21,21
4,"""Le prof d'EPS qui donne des cours de langue ?...",meli ʋliʋlim xena fetu sɔsɔe xɔxɔ le dɔ ƒomevi...,https://ellecitoyenne.com/,"Blog, News",45,11


In [7]:
# On garde la même taille que dans le notebook 06 pour comparer correctement.
# Si Colab tient bien, on pourra augmenter plus tard.

TRAIN_SAMPLE_SIZE = 3000
VALID_SAMPLE_SIZE = 300
TEST_SAMPLE_SIZE = 200

train_small = train_df.sample(TRAIN_SAMPLE_SIZE, random_state=42).reset_index(drop=True)
valid_small = valid_df.sample(VALID_SAMPLE_SIZE, random_state=42).reset_index(drop=True)
test_small = test_df.sample(TEST_SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print("Train small :", train_small.shape)
print("Valid small :", valid_small.shape)
print("Test small :", test_small.shape)

Train small : (3000, 6)
Valid small : (300, 6)
Test small : (200, 6)


In [8]:
# Pour le fine-tuning éwé → français :
# - source = phrase en éwé
# - target = phrase en français

train_small = train_small[["ewe", "french"]].dropna()
valid_small = valid_small[["ewe", "french"]].dropna()
test_small = test_small[["ewe", "french"]].dropna()

train_small.head()

,ewe,french
0,bubuwo mateŋjaƒle tampɔ̃ o esiatae wole aƒelme...,D'autres ne peuvent pas s'acheter des tampons ...
1,nuwuwua ɖee fia be « John Ziguéhi » ɖo eƒe ŋug...,"En lisant la fin, on suppose que « John Ziguéh..."
2,eƒo ga adẽ kple afã eye medo le dɔa me,Il est 18h30 lorsque je descends enfin du boulot
3,Afrika ame ŋkuta gã siawo abe nye Chimamand...,Le discours des grandes figures africaines du ...
4,lɔlɔ̃tɔe la elɔ̃ ɖe eƒe asiliameŋu kple dzidzi...,"Amoureuse, elle a cédé à ses avances, à ses ba..."


In [9]:
train_dataset = Dataset.from_pandas(train_small)
valid_dataset = Dataset.from_pandas(valid_small)
test_dataset = Dataset.from_pandas(test_small)

train_dataset

Dataset({
    features: ['ewe', 'french'],
    num_rows: 3000
})

In [10]:
model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("NLLB chargé.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

NLLB chargé.


In [11]:
# Codes NLLB :
# Éwé      : ewe_Latn
# Français : fra_Latn

EWE_CODE = "ewe_Latn"
FRENCH_CODE = "fra_Latn"

tokenizer.src_lang = EWE_CODE

base_model.generation_config.forced_bos_token_id = tokenizer.convert_tokens_to_ids(FRENCH_CODE)

print("Langue source :", EWE_CODE)
print("Langue cible :", FRENCH_CODE)
print("forced_bos_token_id :", base_model.generation_config.forced_bos_token_id)

Langue source : ewe_Latn
Langue cible : fra_Latn
forced_bos_token_id : 256057


In [12]:
# LoRA permet d'adapter NLLB sans entraîner tous les paramètres.
# On entraîne seulement un petit nombre de paramètres ajoutés au modèle.

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(base_model, lora_config)

# On applique aussi la configuration de génération au modèle LoRA.
model.generation_config.forced_bos_token_id = tokenizer.convert_tokens_to_ids(FRENCH_CODE)

model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 616,253,440 || trainable%: 0.1914


In [13]:
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

In [14]:
def preprocess_function(examples):
    # Langue source : éwé
    tokenizer.src_lang = EWE_CODE

    # Entrée du modèle : phrases en éwé
    inputs = examples["ewe"]

    # Sortie attendue : phrases françaises
    targets = examples["french"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [15]:
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=valid_dataset.column_names
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

tokenized_train

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})

In [16]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [17]:
sacrebleu_metric = evaluate.load("sacrebleu")

In [18]:
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels


def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = sacrebleu_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {"bleu": result["score"]}

In [19]:
output_dir = os.path.join(models_dir, "nllb_ewe_to_fr_lora")

In [20]:
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,

    num_train_epochs=3,
    learning_rate=2e-4,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,

    # Compatible avec ta version récente de transformers.
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

In [21]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [22]:
# On lance le fine-tuning NLLB éwé → français.

trainer.train()

Epoch,Training Loss,Validation Loss,Bleu
1,23.456970,2.572150,6.489546
2,21.778064,2.490856,6.978000
3,21.082087,2.475819,6.945103


TrainOutput(global_step=564, training_loss=23.669774319263215, metrics={'train_runtime': 1094.8208, 'train_samples_per_second': 8.221, 'train_steps_per_second': 0.515, 'total_flos': 479189888335872.0, 'train_loss': 23.669774319263215, 'epoch': 3.0})

In [23]:
test_results = trainer.evaluate(tokenized_test)

test_results

Training Loss,Validation Loss,Epoch,Bleu
21.082087,2.488972,3,7.417988


{'eval_loss': 2.4889724254608154, 'eval_bleu': 7.417987860832811}

In [24]:
finetuned_model_dir = os.path.join(models_dir, "nllb_ewe_to_fr_lora_final")

trainer.save_model(finetuned_model_dir)
tokenizer.save_pretrained(finetuned_model_dir)

print("Modèle fine-tuné sauvegardé ici :", finetuned_model_dir)

Modèle fine-tuné sauvegardé ici : /content/drive/MyDrive/french_ewe_project/models/nllb_ewe_to_fr_lora_final


In [25]:
def translate_finetuned_nllb_ewe_to_fr(text, max_length=128):
    tokenizer.src_lang = EWE_CODE

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(model.device)

    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(FRENCH_CODE),
        max_length=max_length,
        num_beams=4
    )

    translation = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

    return translation

In [26]:
for i in range(5):
    ewe_sentence = test_small.iloc[i]["ewe"]
    true_french = test_small.iloc[i]["french"]

    predicted_french = translate_finetuned_nllb_ewe_to_fr(ewe_sentence)

    print("Éwé original       :", ewe_sentence)
    print("Français attendu   :", true_french)
    print("Français fine-tuné :", predicted_french)
    print("-" * 100)

Éwé original       : Esi doglo bi nyuie la, Mariko dze eɖuɖu.
Français attendu   : Quand le lézard fut bien cuit, Mariko, se mit à le manger.
Français fine-tuné : Une fois qu'elle était bien préparée, Mariko s'est mise à la manger.
----------------------------------------------------------------------------------------------------
Éwé original       : Medoe yi takpekpe sia elabe enye moslemtɔwo ƒe ɖoɖo
Français attendu   : Je l'ai porté à  cette convention parce que c'était un évènement islamique,
Français fine-tuné : J'ai participé à cette conférence parce que c'était un programme musulman
----------------------------------------------------------------------------------------------------
Éwé original       : metrɔ va asadzi afi si edze kɔtɛ be enɔ te nɔm kpɔm le lae
Français attendu   :  Je suis revenu dans la salle de séjour, où tu avais l’air de m’attendre
Français fine-tuné : Je suis retourné à l'endroit où il était évident qu'il était en train de me regarder
---------------------

In [27]:
phrases_ewe = [
    "Meda akpe",
    "Mele dɔ lém",
    "Meyina aƒe me",
    "Nuka wɔm nèle ?"
]

for phrase in phrases_ewe:
    traduction = translate_finetuned_nllb_ewe_to_fr(phrase)

    print("Éwé :", phrase)
    print("Français fine-tuné :", traduction)
    print("-" * 80)

Éwé : Meda akpe
Français fine-tuné : Je vous remercie
--------------------------------------------------------------------------------
Éwé : Mele dɔ lém
Français fine-tuné : Je suis malade
--------------------------------------------------------------------------------
Éwé : Meyina aƒe me
Français fine-tuné : Je rentre à la maison
--------------------------------------------------------------------------------
Éwé : Nuka wɔm nèle ?
Français fine-tuné : Qu'est-ce que tu fais ?
--------------------------------------------------------------------------------


In [28]:
results_finetuning = pd.DataFrame({
    "model": ["NLLB LoRA fine-tuned"],
    "direction": ["Éwé → Français"],
    "train_samples": [TRAIN_SAMPLE_SIZE],
    "valid_samples": [VALID_SAMPLE_SIZE],
    "test_samples": [TEST_SAMPLE_SIZE],
    "test_bleu": [test_results["eval_bleu"]],
    "test_loss": [test_results["eval_loss"]]
})

results_path = os.path.join(reports_dir, "nllb_finetuning_ewe_to_fr_results.csv")
results_finetuning.to_csv(results_path, index=False)

results_finetuning

,model,direction,train_samples,valid_samples,test_samples,test_bleu,test_loss
0,NLLB LoRA fine-tuned,Éwé → Français,3000,300,200,7.417988,2.488972


## Conclusion

Dans ce notebook, nous avons réalisé un fine-tuning léger de NLLB pour la traduction éwé → français.

Cette étape complète le fine-tuning précédent réalisé dans le sens français → éwé. Le système dispose maintenant d’une approche avancée dans les deux directions de traduction.

Comme dans le notebook précédent, nous avons utilisé LoRA afin d’adapter NLLB sans entraîner tous ses paramètres. Cette méthode permet de réduire la mémoire nécessaire et de rendre le fine-tuning possible dans Google Colab.

Les résultats obtenus seront comparés à ceux du modèle NLLB pré-entraîné et aux baselines LSTM afin d’évaluer l’intérêt du fine-tuning sur notre corpus français-éwé.

## Résultats obtenus

Le fine-tuning LoRA de NLLB pour la traduction éwé → français a été réalisé sur un sous-ensemble du corpus :

- 3000 phrases d'entraînement ;
- 300 phrases de validation ;
- 200 phrases de test.

Les résultats obtenus sur le jeu de test sont :

- Loss : 2.4890
- BLEU : 7.4180

Ce score est supérieur au score obtenu avec NLLB pré-entraîné sans fine-tuning dans le même sens de traduction. Cela montre que l’adaptation du modèle au corpus éwé-français améliore la qualité des traductions.

## Analyse qualitative

Les exemples générés montrent une nette amélioration par rapport aux modèles Seq2Seq LSTM. Le modèle fine-tuné parvient à traduire correctement plusieurs phrases simples comme “Meda akpe”, “Mele dɔ lém” ou “Meyina aƒe me”.

Sur les phrases issues du jeu de test, les traductions ne sont pas toujours identiques aux références, mais elles conservent souvent le sens général. Cela confirme l’intérêt d’utiliser un modèle multilingue pré-entraîné, puis de l’adapter au corpus français-éwé avec LoRA.